<a href="https://colab.research.google.com/github/somendrew/Fine-tuning_LLMs/blob/main/Lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# LoRA (Low-Rank Adaptation)

---

## 1️⃣ Full Form

**LoRA = Low-Rank Adaptation**

It is a **parameter-efficient fine-tuning (PEFT)** technique used to adapt Large Language Models (LLMs).

---

## 2️⃣ Core Idea

Instead of updating the full weight matrix \( W \), LoRA:

- ❄️ Freezes original pretrained weights  
- ➕ Adds a small learnable update  
- 🧠 Trains only small low-rank matrices  

Mathematically:

\[
W' = W + \Delta W
\]

Instead of learning full \( \Delta W \), LoRA factorizes it:

\[
\Delta W = A \times B
\]

So final weight:

\[
W' = W + A B
\]

Only **A and B are trainable**.

---

## 3️⃣ Why “Low-Rank”?

If original weight:

```

4096 × 4096 ≈ 16M parameters

```

LoRA with rank = 8:

```

4096×8 + 8×4096 ≈ 65K parameters

````

Huge reduction.

Low-rank means:

> Instead of learning a full big matrix, we approximate it using two small matrices.

---

## 4️⃣ Simple Intuition

Think of it like this:

- Full fine-tuning → Rewrite the whole book 📖  
- LoRA → Add small sticky notes that slightly adjust meaning 📝  

The model already knows a lot — it only needs small adjustments.

---

## 5️⃣ Why LoRA Saves Memory

- Optimizer states (Adam moments) are stored only for A and B
- Original weights are frozen
- GPU memory usage drops massively
- Enables fine-tuning large LLMs on smaller GPUs

---

## 6️⃣ Where LoRA is Applied in Transformers

LoRA is usually applied to:

- Query (Q) projection
- Key (K) projection
- Value (V) projection
- Sometimes Output (O) projection

Mostly inside the **attention layers**.

---

## 7️⃣ TensorFlow Implementation (Clean Version)

```python
import tensorflow as tf

class LoRALinear(tf.keras.layers.Layer):
    def __init__(self, units, rank=8):
        super().__init__()
        self.units = units
        self.rank = rank

    def build(self, input_shape):
        input_dim = input_shape[-1]

        # Frozen original weight
        self.W = self.add_weight(
            shape=(input_dim, self.units),
            initializer="random_normal",
            trainable=False,
            name="W"
        )

        # Trainable low-rank matrices
        self.A = self.add_weight(
            shape=(input_dim, self.rank),
            initializer="random_normal",
            trainable=True,
            name="A"
        )

        self.B = self.add_weight(
            shape=(self.rank, self.units),
            initializer="zeros",
            trainable=True,
            name="B"
        )

    def call(self, inputs):
        delta_W = tf.matmul(self.A, self.B)
        return tf.matmul(inputs, self.W + delta_W)
````

---

## 8️⃣ How to Use It in a Model

```python
model = tf.keras.Sequential([
    LoRALinear(128, rank=4),
    tf.keras.layers.ReLU(),
    LoRALinear(10, rank=4)
])

model.compile(optimizer="adam", loss="mse")
```

Only **A and B are updated during training**.

---

## 9️⃣ Interview-Ready One-Liner

> LoRA fine-tunes large language models by freezing original weights and learning a low-rank matrix decomposition (A×B) that approximates weight updates, drastically reducing memory and training cost.

```
```
